# Phan tich Du lieu

In [1]:
import os
import re
import pandas as pd
import numpy as np
import networkx as nx
from tqdm import tqdm
from pyvi import ViTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter

In [2]:
# === CONFIG ===
DOCS_PATH = "../1.CollectingDocuments/data_clean"
LINK_FILE = "../1.CollectingDocuments/extracted_urls.csv"

## Doc tai lieu

In [3]:
# === Đọc dữ liệu và tiền xử lý ===
def clean_text(text):
    # Loại bỏ ký tự đặc biệt, giữ lại tiếng Việt có dấu
    text = re.sub(r"http\S+", "", text)  
    text = re.sub(r"\s+", " ", text).strip()
    return text

def load_and_preprocess_docs(folder_path):
    docs = {}
    for fname in os.listdir(folder_path):
        if fname.endswith(".txt"):
            with open(os.path.join(folder_path, fname), "r", encoding="utf-8") as f:
                raw = f.read()
            clean = clean_text(raw)
            tokens = ViTokenizer.tokenize(clean)
            docs[fname] = tokens
    return docs

print("🔄 Đang tải và xử lý tài liệu...")
docs = load_and_preprocess_docs(DOCS_PATH)
print(f"✅ Đã xử lý {len(docs)} tài liệu.\n")

🔄 Đang tải và xử lý tài liệu...
✅ Đã xử lý 494 tài liệu.



In [4]:
# Thêm mới
# Thu thập tất cả tokens
all_tokens = []
doc_token_lists = {}  # Lưu list tokens của mỗi doc để dùng sau
for fname, text in docs.items():
    tokens = text.split()
    doc_token_lists[fname] = tokens
    all_tokens.extend(tokens)

# Thống kê tổng quan
unique_terms_global = len(set(all_tokens))
total_tokens_global = len(all_tokens)

print(f"\n THỐNG KÊ TỔNG QUAN:")
print(f"   ├─ Tổng số documents: {len(docs)}")
print(f"   ├─ Tổng số tokens: {total_tokens_global:,}")
print(f"   ├─ Số terms duy nhất (vocabulary): {unique_terms_global:,}")
print(f"   └─ Tỷ lệ unique/total: {unique_terms_global/total_tokens_global*100:.1f}%")

# Độ dài documents
doc_lengths = [len(tokens) for tokens in doc_token_lists.values()]
avgdl = np.mean(doc_lengths)
min_len = np.min(doc_lengths)
max_len = np.max(doc_lengths)
std_len = np.std(doc_lengths)

print(f"\n ĐỘ DÀI DOCUMENTS:")
print(f"   ├─ Độ dài trung bình (avgdl): {avgdl:.1f} terms")
print(f"   ├─ Độ dài min: {min_len} terms")
print(f"   ├─ Độ dài max: {max_len} terms")
print(f"   └─ Độ lệch chuẩn: {std_len:.1f}")



 THỐNG KÊ TỔNG QUAN:
   ├─ Tổng số documents: 494
   ├─ Tổng số tokens: 1,339,696
   ├─ Số terms duy nhất (vocabulary): 39,071
   └─ Tỷ lệ unique/total: 2.9%

 ĐỘ DÀI DOCUMENTS:
   ├─ Độ dài trung bình (avgdl): 2711.9 terms
   ├─ Độ dài min: 14 terms
   ├─ Độ dài max: 21657 terms
   └─ Độ lệch chuẩn: 2426.3


In [5]:
# Thêm mới 
corpus_stats_summary = {
    'total_documents': len(docs),
    'total_tokens': total_tokens_global,
    'unique_terms': unique_terms_global,
    'avgdl': avgdl,
    'min_length': min_len,
    'max_length': max_len,
    'std_length': std_len,
}

df_corpus_summary = pd.DataFrame([corpus_stats_summary])
df_corpus_summary.to_csv("output/corpus_summary.csv", index=False, encoding="utf-8-sig")
print("Đã lưu tổng hợp corpus: output/corpus_summary.csv")

Đã lưu tổng hợp corpus: output/corpus_summary.csv


In [6]:
# Thêm mới
term_doc_count = Counter()
for tokens in doc_token_lists.values():
    unique_in_doc = set(tokens)
    for term in unique_in_doc:
        term_doc_count[term] += 1

print(f"\nTOP 20 TERMS (Document Frequency):")
top_20_terms = term_doc_count.most_common(20)
for i, (term, df) in enumerate(top_20_terms, 1):
    pct = df / len(docs) * 100
    idf = np.log(len(docs) / df)
    print(f"   {i:2d}. {term:30s} | DF={df:4d} ({pct:5.1f}%) | IDF={idf:.3f}")


TOP 20 TERMS (Document Frequency):
    1. .                              | DF= 492 ( 99.6%) | IDF=0.004
    2. ,                              | DF= 453 ( 91.7%) | IDF=0.087
    3. :                              | DF= 451 ( 91.3%) | IDF=0.091
    4. của                            | DF= 448 ( 90.7%) | IDF=0.098
    5. và                             | DF= 448 ( 90.7%) | IDF=0.098
    6. trong                          | DF= 447 ( 90.5%) | IDF=0.100
    7. đến                            | DF= 447 ( 90.5%) | IDF=0.100
    8. -                              | DF= 446 ( 90.3%) | IDF=0.102
    9. được                           | DF= 446 ( 90.3%) | IDF=0.102
   10. có                             | DF= 444 ( 89.9%) | IDF=0.107
   11. là                             | DF= 443 ( 89.7%) | IDF=0.109
   12. để                             | DF= 441 ( 89.3%) | IDF=0.113
   13. các                            | DF= 440 ( 89.1%) | IDF=0.116
   14. trên                           | DF= 439 ( 88.9%) | IDF=0.11

In [ ]:
# Thêm mới
df_top_terms = pd.DataFrame(top_20_terms, columns=['term', 'document_frequency'])
df_top_terms['percentage'] = (df_top_terms['document_frequency'] / len(docs) * 100).round(1)
df_top_terms['idf'] = np.log(len(docs) / df_top_terms['document_frequency']).round(3)
df_top_terms.to_csv("output/top_terms_by_df.csv", index=False, encoding="utf-8-sig")
print("Đã lưu top terms: output/top_terms_by_df.csv")

✅ Đã lưu top terms: output/top_terms_by_df.csv


In [ ]:
# === Phân tích thống kê cơ bản ===
doc_stats = []
for fname, text in docs.items():
    tokens = text.split()
    total_words = len(tokens)
    unique_words = len(set(tokens))
    top_words = [w for w, _ in Counter(tokens).most_common(10)]
    doc_stats.append({
        "document": fname,
        "total_words": total_words,
        "unique_words": unique_words,
        "top_words": ", ".join(top_words)
    })

df_stats = pd.DataFrame(doc_stats)
print("📊 Thống kê cơ bản:\n", df_stats.head(), "\n")


📊 Thống kê cơ bản:
                                         document  total_words  unique_words  \
0        Cam_nang_du_lich_Tam_Coc_-_Bich_ong.txt         1819           777   
1             Cam_nang_du_lich_Chua_Tam_Chuc.txt         2004           737   
2                     Cam_nang_du_lich_Ky_Co.txt         1433           589   
3                   Cam_nang_du_lich_Bao_Loc.txt         2152           784   
4  oi_che_Moc_Chau_-_iem_en_hap_dan_du_khach.txt         2332           745   

                                           top_words  
0  có, đồng, tam_cốc, là, những, lúa, ninh, xem, ...  
1  tam, chúc, chùa, điện, có, đến, là, du_lịch, v...  
2         co, kỳ, đi, nhơn, là, có, ở, biển, quy, để  
3  lộc, bảo, là, và, với, có, đồng, du_khách, thà...  
4  10, 2025, ngày, mộc, châu, chè, của, và, mới, ...   



In [ ]:
# === TF-IDF cho toàn bộ tập tài liệu ===
vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = vectorizer.fit_transform(docs.values())
feature_names = vectorizer.get_feature_names_out()

In [10]:
# Trích 5 từ có TF-IDF cao nhất cho mỗi doc
def top_tfidf_words(row, features, top_n=5):
    idx = row.nonzero()[1]
    scores = zip(idx, [row[0, i] for i in idx])
    sorted_words = sorted(scores, key=lambda x: x[1], reverse=True)[:top_n]
    return ", ".join([features[i] for i, _ in sorted_words])

top_tfidf = [top_tfidf_words(tfidf_matrix[i], feature_names) for i in range(len(docs))]
df_stats["top_tfidf"] = top_tfidf

In [11]:
df_stats.head()

,document,total_words,unique_words,top_words,top_tfidf
0,Cam_nang_du_lich_Tam_Coc_-_Bich_ong.txt,1819,777,"có, đồng, tam_cốc, là, những, lúa, ninh, xem, ...","tam_cốc, lúa, hang, ninh, đò"
1,Cam_nang_du_lich_Chua_Tam_Chuc.txt,2004,737,"tam, chúc, chùa, điện, có, đến, là, du_lịch, v...","chúc, tam, chùa, điện, thủy_đình"
2,Cam_nang_du_lich_Ky_Co.txt,1433,589,"co, kỳ, đi, nhơn, là, có, ở, biển, quy, để","co, nhơn, kỳ, quy, đi"
3,Cam_nang_du_lich_Bao_Loc.txt,2152,784,"lộc, bảo, là, và, với, có, đồng, du_khách, thà...","lộc, bảo, dambri, chè, là"
4,oi_che_Moc_Chau_-_iem_en_hap_dan_du_khach.txt,2332,745,"10, 2025, ngày, mộc, châu, chè, của, và, mới, ...","chè, chào, mộc, 2025, châu"


In [ ]:
# # === Phân tích liên kết & tính PageRank ===
# if os.path.exists(LINK_FILE):
#     links = pd.read_csv(LINK_FILE)
#     G = nx.DiGraph()
#     for _, row in links.iterrows():
#         src, tgt = row["source_file"], row["target"]
#         if src in docs and tgt in docs:
#             G.add_edge(src, tgt)
    
#     # Tính PageRank (độ phổ biến / ảnh hưởng)
#     pagerank_scores = nx.pagerank(G, alpha=0.85)
#     df_stats["pagerank"] = df_stats["document"].map(pagerank_scores).fillna(0)
# else:
#     df_stats["pagerank"] = 0
#     print("⚠️ Không tìm thấy file liên kết, bỏ qua PageRank.\n")

In [ ]:
# === Chuẩn hóa dữ liệu tổng hợp ===
df_stats["avg_tfidf"] = tfidf_matrix.mean(axis=1).A1
# df_stats = df_stats.sort_values(ascending=False)

In [13]:
print("🏁 Kết quả tổng hợp:")
print(df_stats.head())

🏁 Kết quả tổng hợp:
                                        document  total_words  unique_words  \
0        Cam_nang_du_lich_Tam_Coc_-_Bich_ong.txt         1819           777   
1             Cam_nang_du_lich_Chua_Tam_Chuc.txt         2004           737   
2                     Cam_nang_du_lich_Ky_Co.txt         1433           589   
3                   Cam_nang_du_lich_Bao_Loc.txt         2152           784   
4  oi_che_Moc_Chau_-_iem_en_hap_dan_du_khach.txt         2332           745   

                                           top_words  \
0  có, đồng, tam_cốc, là, những, lúa, ninh, xem, ...   
1  tam, chúc, chùa, điện, có, đến, là, du_lịch, v...   
2         co, kỳ, đi, nhơn, là, có, ở, biển, quy, để   
3  lộc, bảo, là, và, với, có, đồng, du_khách, thà...   
4  10, 2025, ngày, mộc, châu, chè, của, và, mới, ...   

                          top_tfidf  avg_tfidf  
0      tam_cốc, lúa, hang, ninh, đò   0.003778  
1  chúc, tam, chùa, điện, thủy_đình   0.002559  
2             co, nhơ

In [ ]:
# === Xuất ra file tổng hợp ===
os.makedirs("output", exist_ok=True)
df_stats.to_csv("output/doc_analysis.csv", index=False, encoding="utf-8-sig")
print("\n💾 Đã lưu kết quả vào output/doc_analysis.csv")


💾 Đã lưu kết quả vào output/doc_analysis.csv
